In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df19 = pd.read_excel("national_M2019_dl.xlsx")
df24 = pd.read_excel("national_M2024_dl.xlsx")

df19.columns = [c.lower().strip() for c in df19.columns]
df24.columns = [c.lower().strip() for c in df24.columns]

keep_titles = [
    "All Occupations",
    "Medical and Health Services Managers",
    "Compliance Officers",
    "Information Security Analysts",
    "Computer and Information Systems Managers"
]

def prep(df, year):
    out = df.copy()
    out["area_title"] = out["area_title"].astype(str).str.strip()
    out["i_group"] = out["i_group"].astype(str)

    out = out[(out["area_title"] == "U.S.") & (out["i_group"].str.contains("cross", case=False, na=False))]
    out = out[out["occ_title"].isin(keep_titles)]
    out = out[["occ_code", "occ_title", "a_mean"]].drop_duplicates()

    out["year"] = year
    out["a_mean"] = pd.to_numeric(out["a_mean"], errors="coerce")
    return out

d = pd.concat([prep(df19, 2019), prep(df24, 2024)], ignore_index=True).dropna(subset=["a_mean"])

plt.figure()
for title in keep_titles:
    s = d[d["occ_title"] == title].sort_values("year")
    if len(s) >= 2:
        plt.plot(s["year"], s["a_mean"], marker="o")

plt.title("OEWS Mean Annual Wage (U.S.): 2019 vs 2024")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

plt.figure()

for title in keep_titles:
    s = d[d["occ_title"] == title].sort_values("year")
    if len(s) >= 2:
        plt.plot(s["year"], s["a_mean"], marker="o")

plt.title("OEWS Mean Annual Wage (U.S.): 2019 vs 2024")
plt.xlabel("Year")
plt.ylabel("Mean Annual Wage ($)")
plt.xticks([2019, 2024])
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.legend(keep_titles, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
w = d.pivot_table(index="occ_title", columns="year", values="a_mean", aggfunc="first")
w["pct_change_2019_2024"] = (w[2024] - w[2019]) / w[2019] * 100
w.loc[keep_titles].sort_values("pct_change_2019_2024", ascending=False)